# 🦜🔗 GraphRAG with LangChain + Neo4j
## Implementation Plan — PDF-based Knowledge Graph RAG Pipeline

**Framework**: LangChain + LangGraph + Neo4j  
**Version Standard**: 2026 (All latest stable libraries)  
**Purpose**: Load a PDF → Extract Knowledge Graph → Hybrid Retrieval → LLM Q&A

---

## 📐 Architecture Overview

```
                      ┌─────────────────────────────────────────┐
                      │         LANGCHAIN GRAPHRAG PIPELINE      │
                      └─────────────────────────────────────────┘

  PDF File
     │
     ▼
  PyMuPDF / PyPDF Loader
     │
     ▼
  RecursiveCharacterTextSplitter (chunk_size=512, overlap=50)
     │
     ├───────────────────────────────────────┐
     ▼                                       ▼
  OpenAI Embeddings                  LLMGraphTransformer
  (text-embedding-3-small)            (GPT-4o-mini for extraction)
     │                                       │
     ▼                                       ▼
  Neo4j Vector Index              Neo4j Property Graph
  (Hybrid: vector + keyword)       (Nodes + Relationships)
     │                                       │
     └───────────────────┬───────────────────┘
                         ▼
               EnsembleRetriever
               (Graph + Vector merged)
                         │
                         ▼
                  LangGraph Router
                  ├── Simple → Vector path
                  └── Complex → Cypher path
                         │
                         ▼
                  GPT-4o (Answer Generation)
                         │
                         ▼
                  Final Answer + Graph Citations
```

---

## 📦 Section 1: Environment Setup & Library Installation

In [1]:
# ============================================================
# CELL 1: Install all dependencies (2026 latest versions)
# ============================================================

# Core LangChain ecosystem
# %pip install langchain
# %pip install langchain-community
# %pip install langchain-neo4j       # Official Neo4j LangChain integration
# %pip install langchain-openai
# %pip install langchain-experimental  # LLMGraphTransformer
# %pip install langgraph             # Agentic workflow engine

# # Graph Database
# %pip install neo4j>=5.27.0                   # Neo4j Python driver

# # PDF Processing
# %pip install pymupdf>=1.25.0                 # Fast PDF loader (fitz)
# %pip install pypdf>=4.0.0                    # Alternative PDF loader
# %pip install unstructured[all-docs]  # Layout-aware parsing

# # Graph Visualization
# %pip install yfiles_jupyter_graphs           # Interactive graph in Jupyter
# %pip install networkx                 # Graph algorithms

# # Evaluation
# %pip install ragas                   # RAG evaluation
# %pip install deepeval                # LLM evaluation

# # Utilities
# %pip install python-dotenv
# %pip install tiktoken
# %pip install json-repair
# %pip install tqdm

print("✅ All dependencies installed successfully")

✅ All dependencies installed successfully


### Environment Variables & Configuration

In [20]:

import os
from dotenv import load_dotenv

load_dotenv()  # Load from .env file

# --- API Keys ---
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "sk-your-key-here")

# --- Neo4j Connection ---
# Option A: Local Neo4j Desktop / Docker
os.environ["NEO4J_URI"]      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
os.environ["NEO4J_USERNAME"] = os.getenv("NEO4J_USERNAME", "neo4j")
os.environ["NEO4J_PASSWORD"] = os.getenv("NEO4J_PASSWORD", "password")

# Option B: Neo4j AuraDB (Cloud - recommended for production)
# os.environ["NEO4J_URI"] = "neo4j+s://xxxxxxxx.databases.neo4j.io"

# --- Configuration ---
PDF_PATH       = "data/Understanding_Climate_Change.pdf"   # ← Set your PDF path here
CHUNK_SIZE     = 1024
CHUNK_OVERLAP  = 200
EMBED_MODEL    = "text-embedding-3-small"     # 1536-dim, cost-efficient
LLM_EXTRACT    = "gpt-4o-mini"               # Cheaper model for extraction
LLM_GENERATE   = "gpt-4o"                    # Stronger model for final answer
VECTOR_INDEX   = "pdf_vector_index"
NODE_LABEL     = "Chunk"

print("✅ Configuration loaded")
print(f"   PDF:           {PDF_PATH}")
print(f"   Neo4j URI:     {os.environ['NEO4J_URI']}")
print(f"   Embed Model:   {EMBED_MODEL}")
print(f"   LLM Extract:   {LLM_EXTRACT}")
print(f"   LLM Generate:  {LLM_GENERATE}")

✅ Configuration loaded
   PDF:           data/Understanding_Climate_Change.pdf
   Neo4j URI:     bolt://localhost:7687
   Embed Model:   text-embedding-3-small
   LLM Extract:   gpt-4o-mini
   LLM Generate:  gpt-4o


## 📁 Section 2: PDF Loading & Document Processing

**Strategy**: Use PyMuPDF for fast, layout-preserving extraction. Chunk with overlap to preserve context across boundaries.

In [21]:
from langchain_community.document_loaders.pdf import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH="data/Understanding_Climate_Change.pdf"

# --- Load PDF ---
print(f"Loading PDF: {PDF_PATH}")
loader = PyMuPDFLoader(PDF_PATH)
raw_documents = loader.load()

print(f"✅ Loaded {len(raw_documents)} pages from PDF")
print(f"   Sample snippet: {raw_documents[0].page_content[:200]}...")

# --- Text Splitting ---
# RecursiveCharacterTextSplitter is preferred over TokenTextSplitter
# for better semantic boundary preservation
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],  # Hierarchy of separators
    length_function=len,
)

documents = text_splitter.split_documents(raw_documents)

print(f"✅ Split into {len(documents)} chunks")
print(f"   Average chunk length: {sum(len(d.page_content) for d in documents) // len(documents)} chars")

# NOTE: For complex PDFs with tables/figures, use unstructured loader:
# from langchain_community.document_loaders import UnstructuredPDFLoader
# loader = UnstructuredPDFLoader(PDF_PATH, mode="elements", strategy="hi_res")

Loading PDF: data/Understanding_Climate_Change.pdf
✅ Loaded 33 pages from PDF
   Sample snippet: Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the plane...
✅ Split into 97 chunks
   Average chunk length: 854 chars


## 🕸️ Section 3: Knowledge Graph Construction

**LLMGraphTransformer** uses an LLM to extract entities (nodes) and relationships (edges) from text chunks. This is the core of GraphRAG — transforming unstructured text into structured graph data.

### Initialize Neo4j Graph Connection

In [22]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    enhanced_schema=True,   # Richer schema for Cypher generation
    sanitize=True,          # Clean unsafe characters
    refresh_schema=True,    # Always fetch latest schema
)

print("✅ Connected to Neo4j")
print("Graph Schema:")
print(graph.schema)

✅ Connected to Neo4j
Graph Schema:
Node properties:

Relationship properties:

The relationships:



In [ ]:
# Sample schema definition for LLMGraphTransformer
# graph_transformer = LLMGraphTransformer(
#     llm=llm_extractor,
#     allowed_nodes=[
#         "Person", "Organization", "Concept", "Technology",
#         "Process", "Event", "Location", "Document", "Product"
#     ],
#     allowed_relationships=[
#         "RELATED_TO", "WORKS_AT", "MENTIONS", "PART_OF",
#         "CAUSES", "USES", "DEFINES", "LOCATED_IN", "LEADS_TO"
#     ],
#     node_properties=["description", "source"],    # Extra metadata on nodes
#     relationship_properties=["confidence"],        # Extra metadata on edges
#     strict_mode=False,  # Allow extraction outside schema (set True for strict control)
# )

### Entity & Relationship Extraction with LLMGraphTransformer

In [23]:
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*")
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_openai import ChatOpenAI

# Use cheaper model for extraction (high volume API calls)
llm_extractor = ChatOpenAI(
    model=LLM_EXTRACT,
    temperature=0,
)

# --- Schema-Constrained Extraction ---
# Define domain-specific node types and relationship types
# IMPORTANT: Customize these for YOUR document domain
graph_transformer = LLMGraphTransformer(
     llm=llm_extractor,
     allowed_nodes=[
         "Concept",        # Climate Change, Greenhouse Effect, Carbon Cycle
         "Phenomenon",     # Heatwaves, Droughts, Sea Level Rise, Hurricanes
         "Substance",      # CO2, CH4, N2O, Methane, Carbon Dioxide
         "Process",        # Deforestation, Combustion, Carbon Sequestration
         "Technology",     # Solar Power, Wind Turbine, Fracking
         "Ecosystem",      # Tropical Rainforest, Coral Reef, Boreal Forest
         "Organization",   # IPCC
         "Location",       # Amazon, Arctic, Himalayas, China, India
     ],
     allowed_relationships=[
         "CAUSES",         # Greenhouse Gases CAUSES Climate Change
         "CONTRIBUTES_TO", # Deforestation CONTRIBUTES_TO Greenhouse Effect
         "LEADS_TO",       # Climate Change LEADS_TO Sea Level Rise
         "IMPACTS",        # Climate Change IMPACTS Coral Reefs
         "EMITS",          # Livestock EMITS Methane
         "ABSORBS",        # Forests ABSORBS CO2
         "MITIGATES",      # Solar Power MITIGATES Climate Change
         "PART_OF",        # Coal PART_OF Fossil Fuels
         "LOCATED_IN",     # Amazon LOCATED_IN South America
         "RELATED_TO",     # Generic fallback
     ],
     node_properties=["description", "source"],
     relationship_properties=["confidence"],
     strict_mode=False,
 )


# --- Process documents in batches ---
# NOTE: Process in batches to manage API rate limits
# For large PDFs (100+ pages), consider async batch processing
# BATCH_SIZE = 20
# all_graph_docs = []

# from tqdm import tqdm

# for i in tqdm(range(0, len(documents), BATCH_SIZE), desc="Extracting graph"):
#     batch = documents[i:i+BATCH_SIZE]
#     graph_docs = graph_transformer.convert_to_graph_documents(batch)
#     all_graph_docs.extend(graph_docs)

# print(f"✅ Graph extraction complete")
# print(f"   Processed {len(all_graph_docs)} document chunks")
# print(f"   Sample nodes: {[n.id for n in all_graph_docs[0].nodes[:5]]}")
# print(f"   Sample rels:  {[(r.source.id, r.type, r.target.id) for r in all_graph_docs[0].relationships[:3]]}")

### For Async Processing Only

In [24]:
# uv add nest_asyncio
#%pip install nest_asyncio


import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*")
import asyncio
import nest_asyncio
from tqdm.auto import tqdm

# Required: Jupyter already runs an event loop; nest_asyncio patches it
nest_asyncio.apply()

BATCH_SIZE = 10
MAX_CONCURRENT = 3    # ← Tune this: higher = faster but more rate-limit risk
all_graph_docs = []


async def extract_graph_async(documents):
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    batches = [documents[i:i+BATCH_SIZE] for i in range(0, len(documents), BATCH_SIZE)]
    pbar = tqdm(total=len(batches), desc="Extracting graph (async)")

    async def process_batch(batch):
        async with semaphore:
            result = await graph_transformer.aconvert_to_graph_documents(batch)
            pbar.update(1)
            return result

    results = await asyncio.gather(
        *[process_batch(b) for b in batches],
        return_exceptions=True   # Don't let one failed batch kill everything
    )
    pbar.close()

    all_docs, errors = [], 0
    for i, result in enumerate(results):
        if isinstance(result, Exception):
            print(f"⚠️   Batch {i} failed: {result}")
            errors += 1
        else:
            all_docs.extend(result)

    if errors:
        print(f"⚠️   {errors}/{len(batches)} batches failed — check rate limits or reduce MAX_CONCURRENT")

    return all_docs


all_graph_docs = asyncio.run(extract_graph_async(documents))

Extracting graph (async):   0%|          | 0/10 [00:00<?, ?it/s]

In [25]:
print(len(all_graph_docs))

97


In [26]:
all_graph_docs[0].nodes[:5]

[Node(id='Climate Change', type='Concept', properties={}),
 Node(id='Global Climate', type='Concept', properties={}),
 Node(id='Human Activities', type='Concept', properties={}),
 Node(id='Fossil Fuels', type='Substance', properties={}),
 Node(id='Deforestation', type='Process', properties={})]

In [27]:
all_graph_docs[0].relationships[:5]

[Relationship(source=Node(id='Climate Change', type='Concept', properties={}), target=Node(id='Global Climate', type='Concept', properties={}), type='RELATED_TO', properties={}),
 Relationship(source=Node(id='Human Activities', type='Concept', properties={}), target=Node(id='Climate Change', type='Concept', properties={}), type='CONTRIBUTES_TO', properties={}),
 Relationship(source=Node(id='Fossil Fuels', type='Substance', properties={}), target=Node(id='Climate Change', type='Concept', properties={}), type='CONTRIBUTES_TO', properties={}),
 Relationship(source=Node(id='Deforestation', type='Process', properties={}), target=Node(id='Climate Change', type='Concept', properties={}), type='CONTRIBUTES_TO', properties={}),
 Relationship(source=Node(id="Earth'S Climate", type='Concept', properties={}), target=Node(id='Glacial Cycles', type='Phenomenon', properties={}), type='RELATED_TO', properties={})]

### Store Graph in Neo4j

In [28]:
# Add all extracted graph documents to Neo4j
# include_source=True links each node back to its source document chunk
graph.add_graph_documents(
    all_graph_docs,
    baseEntityLabel=True,   # Adds __Entity__ label for easy global querying
    include_source=True,    # Creates (node)-[:MENTIONED_IN]->(Document) edges
)

# Refresh schema after loading
graph.refresh_schema()

# --- Create Full-Text Index IMMEDIATELY after loading ---
# IMPORTANT: Must exist before GraphRetriever is initialized (used in Section 4)
graph.query("""
    CREATE FULLTEXT INDEX entity_fulltext IF NOT EXISTS
    FOR (n:__Entity__) ON EACH [n.id]
""")
graph.query("CREATE INDEX entity_id IF NOT EXISTS FOR (n:__Entity__) ON (n.id)")
graph.query("CREATE INDEX chunk_text IF NOT EXISTS FOR (n:Document) ON (n.text)")

print("✅ Graph stored in Neo4j")
print("✅ Full-text index 'entity_fulltext' created (required by GraphRetriever)")
print("Updated schema:")
print(graph.schema)

# Quick stats
node_count = graph.query("MATCH (n) RETURN count(n) as nodes")[0]['nodes']
rel_count  = graph.query("MATCH ()-[r]->() RETURN count(r) as rels")[0]['rels']
print(f"\n📊 Graph Stats: {node_count} nodes, {rel_count} relationships")

✅ Graph stored in Neo4j
✅ Full-text index 'entity_fulltext' created (required by GraphRetriever)
Updated schema:
Node properties:
- **Document**
  - `id`: STRING Example: "0e42241b14452a73a228858ae4ec2215"
  - `text`: STRING Example: "Understanding Climate Change  Chapter 1: Introduct"
  - `file_path`: STRING Available options: ['data/Understanding_Climate_Change.pdf']
  - `creator`: STRING Available options: ['Microsoft® Word 2021']
  - `creationdate`: STRING Available options: ['2024-07-13T20:17:34+03:00']
  - `modDate`: STRING Available options: ["D:20240713201734+03'00'"]
  - `keywords`: STRING Available options: ['']
  - `trapped`: STRING Available options: ['']
  - `author`: STRING Available options: ['Nir']
  - `subject`: STRING Available options: ['']
  - `format`: STRING Available options: ['PDF 1.7']
  - `source`: STRING Available options: ['data/Understanding_Climate_Change.pdf']
  - `total_pages`: INTEGER Min: 33, Max: 33
  - `title`: STRING Available options: ['']
  - `cre

## 🔍 Section 4: Vector Index Setup (Hybrid Retrieval)

Neo4j natively supports vector indexes since v5.11+. We configure **hybrid** retrieval (vector + keyword) on the existing graph — no separate vector DB needed.

### Create Hybrid Vector Index on Neo4j

In [29]:
from langchain_neo4j import Neo4jVector
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    dimensions=1536,
)

# Create or connect to existing Neo4j vector index
# from_existing_graph() adds vector embeddings to existing Document nodes
vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    search_type="hybrid",                         # ← Vector + Keyword (BM25)
    node_label="Document",                         # Nodes to embed
    text_node_properties=["text"],                 # Property to embed
    embedding_node_property="embedding",           # Where to store embeddings
    index_name=VECTOR_INDEX,
    keyword_index_name="pdf_keyword_index",        # BM25 keyword index
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
)

print("✅ Neo4j hybrid vector index created")
print(f"   Index name: {VECTOR_INDEX}")
print(f"   Search type: hybrid (vector + BM25 keyword)")

# Quick test
test_results = vector_index.similarity_search("test query", k=2)
print(f"   Test search returned {len(test_results)} results")

✅ Neo4j hybrid vector index created
   Index name: pdf_vector_index
   Search type: hybrid (vector + BM25 keyword)
   Test search returned 2 results


### Graph Retriever (Structured Traversal)

In [30]:
# Full-text entity search → neighborhood traversal
# This retriever finds relevant entities, then fetches their graph neighborhood

from langchain_neo4j import Neo4jGraph
from langchain.schema import BaseRetriever, Document
from langchain_openai import ChatOpenAI
from typing import List
import json

class GraphRetriever(BaseRetriever):
    """Custom graph retriever: Entity recognition → Cypher traversal → Context"""
    
    graph: Neo4jGraph
    llm: ChatOpenAI
    hops: int = 2  # Depth of graph traversal

    def _get_relevant_documents(self, query: str) -> List[Document]:
        # Step 1: Extract key entities from the query
        entity_extraction_prompt = f"""
        Extract the key entities (people, organizations, concepts, technologies) 
        from this question as a JSON list of strings.
        Question: {query}
        Response (JSON list only): 
        """
        entities_raw = self.llm.invoke(entity_extraction_prompt).content
        try:
            entities = json.loads(entities_raw)
        except:
            entities = [query]
        
        # Step 2: Find matching nodes + their neighborhood
        results = []
        for entity in entities[:5]:  # Limit to top 5 entities
            cypher = """
            CALL db.index.fulltext.queryNodes('entity_fulltext', $entity)
            YIELD node, score
            WHERE score > 0.5
            WITH node LIMIT 3
            MATCH path = (node)-[*1..{hops}]-(neighbor)
            RETURN 
                node.id as entity,
                collect(DISTINCT neighbor.id) as neighbors,
                collect(DISTINCT type(relationships(path)[0])) as rel_types
            LIMIT 10
            """.format(hops=self.hops)
            
            try:
                rows = self.graph.query(cypher, {"entity": entity})
                for row in rows:
                    context = f"Entity: {row['entity']}\n"
                    context += f"Connected to: {', '.join(row['neighbors'][:10])}\n"
                    context += f"Via relationships: {', '.join(set(row['rel_types']))}"
                    results.append(Document(page_content=context, 
                                           metadata={"source": "graph", "entity": row['entity']}))
            except Exception as e:
                print(f"  Graph retrieval warning: {e}")
        
        return results


llm_generator = ChatOpenAI(model=LLM_GENERATE, temperature=0)
graph_retriever = GraphRetriever(graph=graph, llm=llm_generator, hops=2)

print("✅ Graph retriever initialized")

✅ Graph retriever initialized


## 🤖 Section 5: LangGraph Agentic Workflow

**LangGraph** adds stateful routing — queries are dynamically routed to the most appropriate retrieval path: graph traversal (complex/relational) or vector search (simple/semantic).

### Build EnsembleRetriever (Graph + Vector)

In [31]:
from langchain.retrievers import EnsembleRetriever

# Create vector retriever
vector_retriever = vector_index.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# Ensemble: 60% weight to graph (structural), 40% to vector (semantic)
# Adjust weights based on your data's relational density
ensemble_retriever = EnsembleRetriever(
    retrievers=[graph_retriever, vector_retriever],
    weights=[0.6, 0.4],
)

print("✅ EnsembleRetriever configured")
print("   Graph weight:  60% (structural reasoning)")
print("   Vector weight: 40% (semantic similarity)")

✅ EnsembleRetriever configured
   Graph weight:  60% (structural reasoning)
   Vector weight: 40% (semantic similarity)


### LangGraph State Machine with Conditional Routing

In [32]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List, Literal
from langchain_core.messages import BaseMessage
import operator

# --- State Definition ---
class GraphRAGState(TypedDict):
    query: str
    query_type: str              # 'simple' | 'complex' | 'relational'
    retrieved_docs: List[str]
    cypher_query: str
    graph_results: str
    final_answer: str
    messages: Annotated[List[BaseMessage], operator.add]

# --- Node Functions ---

def classify_query(state: GraphRAGState) -> GraphRAGState:
    """Classify query as simple (vector) or complex/relational (graph)."""
    query = state["query"]
    classification_prompt = f"""
    Classify this query into ONE category:
    - 'simple': direct fact lookup, definition, simple question
    - 'relational': asks about relationships, connections, 'how X relates to Y'
    - 'complex': multi-hop, thematic, requires traversing multiple concepts
    
    Query: {query}
    Category (one word only):
    """
    result = llm_generator.invoke(classification_prompt).content.strip().lower()
    return {**state, "query_type": result if result in ['simple','relational','complex'] else 'simple'}


def vector_retrieval_node(state: GraphRAGState) -> GraphRAGState:
    """Standard vector similarity search path."""
    docs = vector_retriever.get_relevant_documents(state["query"])
    retrieved = [d.page_content for d in docs]
    return {**state, "retrieved_docs": retrieved}


def graph_retrieval_node(state: GraphRAGState) -> GraphRAGState:
    """Graph traversal retrieval path with Cypher generation."""
    # Generate Cypher query using LLM + schema
    cypher_prompt = f"""
    You are an expert Neo4j Cypher query generator.
    
    Graph Schema:
    {graph.schema}
    
    Generate a Cypher query to answer this question.
    Use MATCH patterns, traverse 1-3 hops, RETURN relevant context.
    Return ONLY the Cypher query, no explanation.
    
    Question: {state['query']}
    Cypher:
    """
    cypher = llm_generator.invoke(cypher_prompt).content.strip()
    
    try:
        graph_results = graph.query(cypher)
        result_str = str(graph_results[:20])  # Limit results
    except Exception as e:
        result_str = f"Graph query failed: {e}. Falling back to vector search."
        docs = vector_retriever.get_relevant_documents(state["query"])
        result_str += "\n" + "\n".join([d.page_content for d in docs])
    
    return {**state, "cypher_query": cypher, "graph_results": result_str}


def hybrid_retrieval_node(state: GraphRAGState) -> GraphRAGState:
    """Hybrid path: ensemble graph + vector retrieval."""
    docs = ensemble_retriever.get_relevant_documents(state["query"])
    retrieved = [d.page_content for d in docs]
    return {**state, "retrieved_docs": retrieved}


def generate_answer(state: GraphRAGState) -> GraphRAGState:
    """Final answer generation with retrieved context."""
    # Combine all available context
    context_parts = []
    if state.get("retrieved_docs"):
        context_parts.append("## Vector + Graph Retrieved Context:\n" + 
                             "\n---\n".join(state["retrieved_docs"]))
    if state.get("graph_results"):
        context_parts.append("## Structured Graph Context:\n" + state["graph_results"])
    
    context = "\n\n".join(context_parts)
    
    answer_prompt = f"""
    You are a helpful AI assistant with access to a knowledge graph and document index.
    Answer the question using ONLY the provided context.
    If the context doesn't contain the answer, say so explicitly.
    Cite your sources when possible.
    
    Context:
    {context}
    
    Question: {state['query']}
    
    Answer:
    """
    answer = llm_generator.invoke(answer_prompt).content
    return {**state, "final_answer": answer}


# --- Routing Function ---
def route_query(state: GraphRAGState) -> Literal["vector", "graph", "hybrid"]:
    query_type = state.get("query_type", "simple")
    if query_type == "simple":
        return "vector"
    elif query_type == "relational":
        return "graph"
    else:  # complex
        return "hybrid"


# --- Build LangGraph ---
workflow = StateGraph(GraphRAGState)

# Add nodes
workflow.add_node("classifier",      classify_query)
workflow.add_node("vector_retrieve", vector_retrieval_node)
workflow.add_node("graph_retrieve",  graph_retrieval_node)
workflow.add_node("hybrid_retrieve", hybrid_retrieval_node)
workflow.add_node("generate",        generate_answer)

# Set entry point
workflow.set_entry_point("classifier")

# Conditional routing after classification
workflow.add_conditional_edges(
    "classifier",
    route_query,
    {
        "vector": "vector_retrieve",
        "graph":  "graph_retrieve",
        "hybrid": "hybrid_retrieve",
    }
)

# All paths converge to generate
workflow.add_edge("vector_retrieve", "generate")
workflow.add_edge("graph_retrieve",  "generate")
workflow.add_edge("hybrid_retrieve", "generate")
workflow.add_edge("generate",        END)

# Compile
graphrag_app = workflow.compile()

print("✅ LangGraph GraphRAG workflow compiled")
print("   Routing logic:")
print("     'simple'     → vector_retrieve → generate")
print("     'relational' → graph_retrieve  → generate")
print("     'complex'    → hybrid_retrieve → generate")

✅ LangGraph GraphRAG workflow compiled
   Routing logic:
     'simple'     → vector_retrieve → generate
     'relational' → graph_retrieve  → generate
     'complex'    → hybrid_retrieve → generate


## 💬 Section 6: Querying the GraphRAG System

### Query Function & Testing

In [37]:
import warnings
warnings.filterwarnings("ignore")

def ask_graphrag(question: str, verbose: bool = True) -> str:
    """Main query interface for the GraphRAG system."""
    print(f"\n{'='*60}")
    print(f"❓ Question: {question}")
    print(f"{'='*60}")
    
    result = graphrag_app.invoke({
        "query": question,
        "query_type": "",
        "retrieved_docs": [],
        "cypher_query": "",
        "graph_results": "",
        "final_answer": "",
        "messages": [],
    })
    
    if verbose:
        print(f"\n🔀 Routing:   '{result['query_type']}' path selected")
        if result.get('cypher_query'):
            print(f"\n📊 Cypher:    {result['cypher_query'][:200]}...")
        print(f"\n✅ Answer:\n{result['final_answer']}")
    
    return result['final_answer']


# --- Example Queries (customize for your PDF content) ---

# Simple vector query
answer1 = ask_graphrag("What are Hydrogen Fuel Cells?")

# Relational/graph query
answer2 = ask_graphrag("How are the impacts of climate change?")

# Complex hybrid query
answer3 = ask_graphrag("What is Agroforestry?")


❓ Question: What are Hydrogen Fuel Cells?

🔀 Routing:   'simple' path selected

✅ Answer:
Hydrogen fuel cells generate electricity by combining hydrogen with oxygen, producing only water as a byproduct. They offer a clean alternative to conventional vehicles, particularly for heavy-duty applications like trucks and buses. Developing a robust hydrogen infrastructure is essential for their success. (Source: Vector + Graph Retrieved Context)

❓ Question: How are the impacts of climate change?

🔀 Routing:   'complex' path selected

✅ Answer:
The impacts of climate change are diverse and significant, affecting various aspects of the environment and human society. Key impacts include:

1. **Extreme Weather Events**: Climate change is linked to an increase in the frequency and severity of extreme weather events, such as hurricanes, heatwaves, droughts, and heavy rainfall. These events can have devastating impacts on communities, economies, and ecosystems.

2. **Rising Temperatures**: Global 

In [36]:
# ============================================================
# CELL 13: Direct Cypher Q&A with GraphCypherQAChain
# ============================================================

from langchain_neo4j import GraphCypherQAChain
from langchain_core.prompts import PromptTemplate

# --- Fixed Cypher Generation Prompt ---
# Root cause of "I don't know": LLM was generating:
#   MATCH (c:Concept {description: 'climate change'})   ← WRONG: uses description + lowercase
# Should be:
#   WHERE toLower(n.id) CONTAINS 'climate change'       ← CORRECT: uses id + case-insensitive

CYPHER_GENERATION_TEMPLATE = """Task: Generate a Cypher statement to query a Neo4j graph database.

CRITICAL Rules — follow exactly:
1. Nodes are identified by their `id` property (e.g., id: 'Climate Change', id: 'CO2')
   - ALWAYS use `id` for lookups: WHERE toLower(n.id) CONTAINS 'keyword'
   - NEVER filter by `description` property — it is long text, not an identifier
2. Use case-insensitive matching for node lookup:
   WHERE toLower(n.id) CONTAINS 'search term'
3. RETURN n.id for entity names (not n.description)
4. Use Neo4j 5+ syntax only — COUNT {{ (n)--() }} not size((n)--())
5. Output raw Cypher ONLY — no backticks, no ```cypher, no explanation

Schema:
{schema}

Correct query examples:
  # Find what causes climate change:
  MATCH (n)-[r:CAUSES|CONTRIBUTES_TO]->(m) WHERE toLower(m.id) CONTAINS 'climate change' RETURN n.id, type(r), m.id LIMIT 10

  # Find impacts of climate change:
  MATCH (n)-[r:IMPACTS|LEADS_TO]->(m) WHERE toLower(n.id) CONTAINS 'climate change' RETURN n.id, type(r), m.id LIMIT 10

  # Most connected nodes:
  MATCH (n) RETURN n.id, COUNT {{ (n)--() }} AS connections ORDER BY connections DESC LIMIT 10

Question: {question}
Cypher Query:"""

CYPHER_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],
    template=CYPHER_GENERATION_TEMPLATE,
)

cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm_generator,
    graph=graph,
    verbose=True,
    validate_cypher=False,         # Off: our prompt handles correctness; validator can corrupt queries
    top_k=10,
    allow_dangerous_requests=True,
    return_intermediate_steps=True,
    cypher_prompt=CYPHER_PROMPT,
)

# Test queries
response = cypher_chain.invoke({"query": "What are the impacts of climate change?"})
print("Answer:", response['result'])
print("Cypher:", response['intermediate_steps'][0]['query'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (n)-[r:IMPACTS|LEADS_TO]->(m) WHERE toLower(n.id) CONTAINS 'climate change' RETURN n.id, type(r), m.id LIMIT 10
Full Context:
[{'n.id': 'Climate Change', 'type(r)': 'IMPACTS', 'm.id': 'Human Activities'}, {'n.id': 'Climate Change', 'type(r)': 'IMPACTS', 'm.id': 'Indigenous Communities'}, {'n.id': 'Climate Change', 'type(r)': 'IMPACTS', 'm.id': 'Effects Of Climate Change'}, {'n.id': 'Climate Change', 'type(r)': 'IMPACTS', 'm.id': 'Ecosystems'}, {'n.id': 'Climate Change', 'type(r)': 'IMPACTS', 'm.id': 'Seasons'}, {'n.id': 'Climate Change', 'type(r)': 'IMPACTS', 'm.id': 'Plant Life Cycles'}, {'n.id': 'Climate Change', 'type(r)': 'IMPACTS', 'm.id': 'Animal Life Cycles'}, {'n.id': 'Climate Change', 'type(r)': 'IMPACTS', 'm.id': 'Agricultural Practices'}, {'n.id': 'Climate Change', 'type(r)': 'IMPACTS', 'm.id': 'International Agreements'}, {'n.id': 'Climate Change', 'type(r)': 'IMPACTS', 'm.id': 'Climate Action'}]

> Finish